# 4 · Designing Agents + Agentic Workflows
Ahora configuramos el agente real de Carrito: objetivos, capacidades y límites.
El checkpoint de preferencias no es una reanudación automática de ejecución.

Predice el resultado antes de ejecutar y anota tus observaciones.

In [ ]:
import json
import os
import sys
from pathlib import Path

# Cada notebook empieza desde datos preparados, sin archivos de sesiones anteriores.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from dotenv import load_dotenv

if not os.getenv("CARRITO_NOTEBOOK_CHECK"):
    load_dotenv(ROOT / ".env")
from carrito.store import create_store
from carrito.tools import StoreTools

db = create_store()
tools = StoreTools(db, user_id="user1")  # Identidad fijada por el host.
RUN_LIVE = False  # Cambiar explícitamente a True permite llamadas de pago.
COMPLETION = {}

def check(name, condition):
    COMPLETION[name] = bool(condition)
    print(("OK" if condition else "PENDIENTE") + ": " + name)

print("MODO OFFLINE: fixtures deterministas. No miden calidad del LLM.")

In [ ]:
from carrito.context import ConversationState
from carrito.lab import AgentProfile, load_state, recover_read, run_profile, save_state
from carrito.model import FixtureModel

profile = AgentProfile(name="shopping", allowed_tools=("search_products", "search_policies"), max_steps=4)
print(profile)
# Decisión: consultar siempre stock/precio en un orden fijo puede ser un workflow.
# Elegir dinámicamente entre aclarar, catálogo y política puede justificar un agente.

## Configurar el loop real
Configura un perfil de lectura. Predice tres recorridos: final normal, tool no permitida y límite de pasos. Cada decisión del fixture está preescrita; el host ejecuta controles reales. Cambiar el prompt no concede permisos.

In [ ]:
def student_profile():
    # TODO: solo search_products; max_steps=2; max_tool_calls=2.
    return AgentProfile(allowed_tools=(), max_steps=1)

In [ ]:
call = {"type": "function_call", "name": "search_products", "call_id": "x", "arguments": json.dumps({"query": "auriculares", "max_price_eur": 80})}
normal = run_profile("Busca", tools, FixtureModel([[call], []]), student_profile())
looping = run_profile("Busca", tools, FixtureModel([[call], [dict(call, call_id="y")], [dict(call, call_id="z")]]), student_profile())
denied_call = {"type": "function_call", "name": "request_return", "call_id": "deny", "arguments": json.dumps({"order_id": "104", "reason": "No encaja"})}
denied = run_profile("Devuelve", tools, FixtureModel([[denied_call], []]), student_profile())
check("recorrido final completo", normal["status"] == "completed")
check("loop acotado", looping["status"] == "step_limit")
check("escritura bloqueada", tools.pending is None)
for name, run in [("normal", normal), ("loop", looping), ("denied", denied)]:
    print(name, run["status"], run["metrics"])
print("Justifica dónde elegirías un workflow fijo en lugar de este loop.")

## Estado y recuperación: qué sobrevive
Guardamos preferencias explícitas, destruimos el objeto en memoria y lo recuperamos.
Esto NO conserva un programa en ejecución ni su call stack. Para retomar una acción hay que comprobar estado de negocio, identidad y confirmación de nuevo.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / "preferences.json"
    state = ConversationState(category="auriculares", budget_eur=80, preferences=["inalámbricos"])
    save_state(path, state)
    del state
    restored = load_state(path)
    print("Estado recuperado:", restored.model_dump())
    resumed_request = run_profile("Continúa la búsqueda", tools, FixtureModel.for_scenario("catalog"), profile, state=restored)
    print(resumed_request["context"])
    # Contraste: el efecto de negocio persiste; la aprobación en memoria no.
    database_path = str(Path(directory) / "shop.sqlite")
    first_store = StoreTools(create_store(database_path))
    first_store.request_return("104", "No encaja")
    first_store.confirm_pending()  # El host representa confirmación del payload mostrado.
    registered = first_store.request_return("104", "No encaja")
    first_store.db.close()
    reopened_store = StoreTools(create_store(database_path))
    repeated = reopened_store.request_return("104", "No encaja")
    assert reopened_store.pending is None and reopened_store.return_count() == 1
    assert repeated["request_id"] == registered["request_id"] and repeated["duplicate"]
    print("Tras reabrir SQLite:", repeated, "pending:", reopened_store.pending)
    reopened_store.db.close()
# Cada solicitud nueva requiere una nueva ejecución: no hay auto-resume.

## Recovery + confirmación
Clasifica errores permanentes y transitorios. Completa una operación de lectura que falle una vez y después consulte SQLite. Compárala con pedido ajeno: ese error no debe reintentarse. Después confirma el payload exacto y repite la escritura; debe existir una sola solicitud.

In [ ]:
attempts = []
def flaky_read():
    # TODO: primer intento temporary_unavailable; siguientes get_order("104").
    attempts.append(1)
    return {"error": "temporary_unavailable"}

In [ ]:
recovered = recover_read(flaky_read, attempts=2)
permanent = recover_read(lambda: tools.get_order("109"), attempts=3)
print("Transitorio:", recovered)
print("Permanente:", permanent)
check("recupera lectura", recovered["result"].get("order_id") == "104")
check("no repite denegación", len(permanent["attempts"]) == 1)
write_tools = StoreTools(create_store())
proposal = write_tools.request_return("104", "No encaja")
assert write_tools.return_count() == 0
write_tools.confirm_pending()  # Clic representado explícitamente por el host.
registered = write_tools.request_return("104", "No encaja")
repeated = write_tools.request_return("104", "No encaja")
assert repeated["request_id"] == registered["request_id"] and write_tools.return_count() == 1
print(proposal, registered, repeated)
# Cambia el motivo entre propuesta y ejecución: vuelve a exigir confirmación.

## Traces y decisiones de diseño
Inspecciona un recorrido correcto y el loop detenido. Señala la primera divergencia, el control que la limita y el dato que necesitas para diagnosticarlo. Un timeout y una denegación necesitan respuestas distintas. No confundir latencia de fixture con rendimiento real del LLM.

In [ ]:
for run in [normal, looping]:
    print(json.dumps(run["events"], ensure_ascii=False, indent=2))
reviews = [{"case": case, "first_divergence": "", "owner": "model/context/tool/host", "change": ""} for case in ["loop", "denied"]]
print(reviews)
if RUN_LIVE:
    from carrito.model import OpenAIModel
    live = run_profile("Busca auriculares por menos de 80 euros", StoreTools(create_store()), OpenAIModel(), profile)
    print(json.dumps(live, ensure_ascii=False, indent=2))

## Salida de sesión
Entrega límites del perfil, dos decisiones workflow/agent y una traza diagnosticada. S5 usa el mismo runtime para configurar, evaluar y mejorar una aplicación completa.

In [ ]:
print(json.dumps(COMPLETION, ensure_ascii=False, indent=2))
print("CHECKPOINT_COMPLETO" if all(COMPLETION.values()) else "Completa las celdas TODO y repite los checks.")